In [2]:
import pandas as pd
import evaluate
import re
import os
import numpy as np
# from rouge_score import rouge_scorer, scoring
# from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple, Any, Sequence
# from transformers import AutoModelForCausalLM, AutoTokenizer,AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import math
from neo4j import GraphDatabase, Transaction 

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default


DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url, echo=False)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS), database=NEO4J_DATABASE)
    

In [3]:
v1_5_system_prompt = """
You are a policy-writing assistant tasked with drafting clear, enforceable cybersecurity and IT governance
policies. Use professional, concise language suitable for formal policy documents.

You may use all provided context—including atomic requirements, ontology definitions, and framework-derived
expectations—to shape the content of the policy. However:

- Do not mention framework names or control identifiers in your output.
- Do not include section numbers or section titles in your output.
- Do not include prefaces, commentary, or meta-text.
- Return only the final policy section text itself.
"""


In [4]:
def generate_v1_5_prompt(
    atomic_requirements: List[str],
    company_ontology: Dict[str, str],
    audience: str = "all employees and contractors",
    section_title: str = "",
    policy_title: str = "",
) -> str:
    """Generates a prompt for the v1.5 policy-writing model."""
    atomic_reqs_text = "\n".join(f"- {req}" for req in atomic_requirements)
    company_ontology_txt = "\n".join(f"- {term}: {definition}" for term, definition in company_ontology.items())
    if len(company_ontology) == 0:
        company_ontology_block = ""
    else:
        company_ontology_block = f"""[COMPANY ONTOLOGY CONTEXT]
The following items describe how this organization defines key operational or governance concepts.
Use them only to ensure alignment with the company environment. Do not restate them verbatim.

{company_ontology_txt}""".strip()

    v1_5_user_prompt = f"""
[INSTRUCTION]
Draft the section titled "{section_title}" for the {policy_title} policy.
Audience: {audience}.
Write only the policy text itself. Tone: professional, direct, and enforceable.

{company_ontology_block}

[ATOMIC REQUIREMENTS]
Below are the granular control expectations that this policy section must operationalize.
Treat these as the minimum set of requirements the policy must clearly support.
Where multiple requirements overlap, you may integrate them naturally into cohesive sentences.

{atomic_reqs_text}

[FRAMEWORK EXPECTATIONS]
These atomic requirements originate from mapped framework controls.
Use them only to shape the policy content.
Do not mention any framework names or control identifiers in the final text.

[CONSTRAINTS]
- Do not include section titles or section numbers.
- Do not include commentary, explanations, or meta text.
- Write as if this text will appear directly in the final approved policy.
"""

    
    return v1_5_user_prompt

In [5]:
def build_v1_5_prompt_df(section_atomic_mapping_df, company_ontology, audience="all employees and contractors"):
    # Sort: direct first, then inferred, preserving atomic order
    df_sorted = (
        section_atomic_mapping_df
        .sort_values(["policy_id", "section_id", "is_direct", "atomic_order"],
                     ascending=[True, True, False, True])
    )

    prompt_rows = []

    for (policy_id, section_id), group in df_sorted.groupby(["policy_id", "section_id"]):
        section_title = group["section_title"].iloc[0]
        policy_title = group["policy_id"].iloc[0]  # or replace with real policy name lookup

        # Build the list of atomic requirements for this section
        atomic_requirements = group["atomic_requirement"].tolist()

        # Generate the V1.5 prompt
        prompt = generate_v1_5_prompt(
            atomic_requirements=atomic_requirements,
            company_ontology=company_ontology,
            audience=audience,
            section_title=section_title,
            policy_title=policy_title,
        )

        prompt_rows.append({
            "policy_id": policy_id,
            "section_id": section_id,
            "section_title": section_title,
            "v1_5_prompt": prompt
        })

    return pd.DataFrame(prompt_rows)


In [6]:
import pandas as pd

allowed = [
    "organization.industry",
    "organization.size",
    "organization.regions",
    "organization.tech_stack",
    "compliance.frameworks",
    "workforce.distribution",
    "data.sensitivity",
    "risk.tolerance",
]

client_ontologies_df = pd.read_sql("SELECT * FROM client_ontologies", engine)

client_ontologies_df = client_ontologies_df[ ~client_ontologies_df["client_id"].isin(["description", "examples"])]

# Keep only what we care about: id + allowed ontology fields
cols_to_keep = ["client_id"] + allowed
client_ontologies_df = client_ontologies_df[cols_to_keep]


In [7]:
section_atomic_mapping_df = pd.read_sql("""SELECT section_atomic_mappings.*, D.policy_title from section_atomic_mappings
JOIN (SELECT DISTINCT policy_id, policy_title FROM policy_lines) d on d.policy_id = section_atomic_mappings.policy_id""", con=engine)
section_atomic_mapping_df['client_id'] = section_atomic_mapping_df['policy_id'].str.split('-').str[0]


In [8]:
section_atomic_mapping_df

,policy_id,section_id,section_title,source_framework,subcontrol_id,atomic_id,atomic_order,atomic_requirement,similarity,is_direct,relation_type,policy_title,client_id
0,CL_0004-CMP-v1.2,CL_0004-CMP-v1.2:S3.5,Administration,SOC2,9292968,637,7,The organization must record and track request...,0.973269,False,inferred,Change Management Policy,CL_0004
1,CL_0004-CMP-v1.2,CL_0004-CMP-v1.2:S3.5,Administration,SOC2,9292946,587,1,The organization must identify and designate c...,0.973396,False,inferred,Change Management Policy,CL_0004
2,CL_0004-CMP-v1.2,CL_0004-CMP-v1.2:S3.5,Administration,SOC2,9292970,623,4,Compliance with privacy objectives must be rev...,0.973523,False,inferred,Change Management Policy,CL_0004
3,CL_0004-CMP-v1.2,CL_0004-CMP-v1.2:S3.5,Administration,SOC2,9292959,667,1,The organization must capture requests for del...,0.973985,False,inferred,Change Management Policy,CL_0004
4,CL_0004-CMP-v1.2,CL_0004-CMP-v1.2:S3.5,Administration,SOC2,9292921,440,11,Management must ensure an employee disciplinar...,0.974108,False,inferred,Change Management Policy,CL_0004
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12885,CL_0005-SDLC-v2.0,CL_0005-SDLC-v2.0:S7,CONTINOUS IMPROVEMENT,NIST CSF,9292389,61,2,The organization must subscribe to vulnerabili...,0.979259,False,inferred,Software Development Lifecycle Policy,CL_0005
12886,CL_0005-SDLC-v2.0,CL_0005-SDLC-v2.0:S7,CONTINOUS IMPROVEMENT,NIST CSF,9292417,224,2,The organization must update system configurat...,0.979583,False,inferred,Software Development Lifecycle Policy,CL_0005
12887,CL_0005-SDLC-v2.0,CL_0005-SDLC-v2.0:S7,CONTINOUS IMPROVEMENT,NIST CSF,9292428,154,2,The organization must define processes for ide...,0.979824,False,inferred,Software Development Lifecycle Policy,CL_0005
12888,CL_0005-SDLC-v2.0,CL_0005-SDLC-v2.0:S7,CONTINOUS IMPROVEMENT,NIST CSF,9292453,318,1,The organization must continually improve even...,0.980536,False,inferred,Software Development Lifecycle Policy,CL_0005


In [9]:
section_atomic_with_onto_df = section_atomic_mapping_df.merge(
    client_ontologies_df,
    on="client_id",      # change if your join key is different
    how="left",
)


In [10]:
import pandas as pd

# Ontology definition table: one row per ontology field (e.g., organization.industry)
ontology_definitions_df = pd.read_sql(
    """
    SELECT
        id,
        label,
        description,
        category,
        seeds,
        counterexamples,
        examples,
        required
    FROM ontology
    """,
    engine,
)

ONTOLOGY_LABELS = (
    ontology_definitions_df
    .set_index("id")["label"]
    .to_dict()
)


In [11]:
def build_company_ontology(row, allowed_keys, label_lookup):
    """
    row: a Series (first row of a group)
    allowed_keys: list of ontology ids we care about (e.g., "organization.industry")
    label_lookup: dict mapping ontology id -> human label (e.g., "Industry")
    Returns: dict[label -> value] for non-empty client-specific values.
    """
    ontology = {}

    for key in allowed_keys:
        if key not in row.index:
            continue

        val = row[key]

        if pd.isna(val):
            continue

        val_str = str(val).strip()
        if not val_str:
            continue

        # Human-friendly label from ontology_definitions; fall back to id if missing
        label = label_lookup.get(key, key)
        ontology[label] = val_str

    return ontology


In [12]:
def build_v1_5_prompt_df(
    section_atomic_with_onto_df,
    allowed_keys,
    audience="all employees and contractors",
    label_lookup=ONTOLOGY_LABELS,
):
    # Sort: direct first, then inferred, preserving atomic order
    df_sorted = (
        section_atomic_with_onto_df
        .sort_values(["policy_id", "section_id", "is_direct", "atomic_order"],
                     ascending=[True, True, False, True])
    )

    prompt_rows = []

    for (policy_id, section_id), group in df_sorted.groupby(["policy_id", "section_id"]):
        section_title = group["section_title"].iloc[0]
        policy_title = group["policy_title"].iloc[0]  # or map to a human-friendly title

        # Build the list of atomic requirements
        atomic_requirements = group["atomic_requirement"].tolist()

        # Build company ontology dict from the first row (same across the group)
        row0 = group.iloc[0]
        company_ontology = build_company_ontology(row0, allowed_keys, label_lookup)


        prompt = generate_v1_5_prompt(
            atomic_requirements=atomic_requirements,
            company_ontology=company_ontology,
            audience=audience,
            section_title=section_title,
            policy_title=policy_title,
        )

        prompt_rows.append({
            "policy_id": policy_id,
            "section_id": section_id,
            "section_title": section_title,
            "v1_5_prompt": prompt,
        })

    return pd.DataFrame(prompt_rows)


In [13]:
section_atomic_with_onto_df[section_atomic_with_onto_df['policy_id']=='CL_0001-ASMP-v1.0']

,policy_id,section_id,section_title,source_framework,subcontrol_id,atomic_id,atomic_order,atomic_requirement,similarity,is_direct,...,policy_title,client_id,organization.industry,organization.size,organization.regions,organization.tech_stack,compliance.frameworks,workforce.distribution,data.sensitivity,risk.tolerance
190,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10.2,"Mobile Devices: Smartphones, Tablets, Etc.",NIST CSF,9292399,97,4,The organization must establish configuration/...,0.968475,False,...,Asset Management Policy,CL_0001,Local Government / Public Administration,Approximately 150 employees (verified via 2024...,"United States – Located in Westchester County,...","Microsoft 365 (Gov tenant), Tyler Technologies...",NIST CSF,Primarily onsite municipal employees across ci...,"Handles PII of residents, employees, and vendo...",Low – public sector environment with strict re...
191,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10.2,"Mobile Devices: Smartphones, Tablets, Etc.",NIST CSF,9292373,17,1,The organization must maintain a list of hardw...,0.968586,True,...,Asset Management Policy,CL_0001,Local Government / Public Administration,Approximately 150 employees (verified via 2024...,"United States – Located in Westchester County,...","Microsoft 365 (Gov tenant), Tyler Technologies...",NIST CSF,Primarily onsite municipal employees across ci...,"Handles PII of residents, employees, and vendo...",Low – public sector environment with strict re...
192,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10.2,"Mobile Devices: Smartphones, Tablets, Etc.",NIST CSF,9292430,235,3,The organization must allow use of nonlocal ma...,0.968884,False,...,Asset Management Policy,CL_0001,Local Government / Public Administration,Approximately 150 employees (verified via 2024...,"United States – Located in Westchester County,...","Microsoft 365 (Gov tenant), Tyler Technologies...",NIST CSF,Primarily onsite municipal employees across ci...,"Handles PII of residents, employees, and vendo...",Low – public sector environment with strict re...
193,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10.2,"Mobile Devices: Smartphones, Tablets, Etc.",NIST CSF,9292398,103,4,System administrators must ensure consoles for...,0.968921,False,...,Asset Management Policy,CL_0001,Local Government / Public Administration,Approximately 150 employees (verified via 2024...,"United States – Located in Westchester County,...","Microsoft 365 (Gov tenant), Tyler Technologies...",NIST CSF,Primarily onsite municipal employees across ci...,"Handles PII of residents, employees, and vendo...",Low – public sector environment with strict re...
194,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10.2,"Mobile Devices: Smartphones, Tablets, Etc.",NIST CSF,9292402,81,2,The organization must document the process for...,0.968927,False,...,Asset Management Policy,CL_0001,Local Government / Public Administration,Approximately 150 employees (verified via 2024...,"United States – Located in Westchester County,...","Microsoft 365 (Gov tenant), Tyler Technologies...",NIST CSF,Primarily onsite municipal employees across ci...,"Handles PII of residents, employees, and vendo...",Low – public sector environment with strict re...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.2,Asset Inventory,NIST CSF,9292402,81,2,The organization must document the process for...,0.971563,False,...,Asset Management Policy,CL_0001,Local Government / Public Administration,Approximately 150 employees (verified via 2024...,"United States – Located in Westchester County,...","Microsoft 365 (Gov tenant), Tyler Technologies...",NIST CSF,Primarily onsite municipal employees across ci...,"Handles PII of residents, employees, and vendo...",Low – public sector environment with strict re...
406,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.2,Asset Inventory,NIST CSF,9292397,106,3,The organization must update the process for c...,0.971751,False,...,Asset Management Policy,CL_0001,Local G

In [14]:
section_atomic_with_onto_df = section_atomic_mapping_df.merge(
    client_ontologies_df,
    on="client_id",
    how="left",
)

v1_5_prompt_df = build_v1_5_prompt_df(
    section_atomic_with_onto_df,
    allowed_keys=allowed,
    label_lookup=ONTOLOGY_LABELS,
)


In [15]:
section_atomic_with_onto_df.iloc[0]

policy_id                                                   CL_0004-CMP-v1.2
section_id                                             CL_0004-CMP-v1.2:S3.5
section_title                                                 Administration
source_framework                                                        SOC2
subcontrol_id                                                        9292968
atomic_id                                                                637
atomic_order                                                               7
atomic_requirement         The organization must record and track request...
similarity                                                          0.973269
is_direct                                                              False
relation_type                                                       inferred
policy_title                                        Change Management Policy
client_id                                                            CL_0004

In [16]:
v1_5_prompt_df

,policy_id,section_id,section_title,v1_5_prompt
0,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S1,PURPOSE,"\n[INSTRUCTION]\nDraft the section titled ""PUR..."
1,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S2,SCOPE,"\n[INSTRUCTION]\nDraft the section titled ""SCO..."
2,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.1,Policy and Procedures,"\n[INSTRUCTION]\nDraft the section titled ""Pol..."
3,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10,Telecommunications and Mobile Devices,"\n[INSTRUCTION]\nDraft the section titled ""Tel..."
4,CL_0001-ASMP-v1.0,CL_0001-ASMP-v1.0:S3.10.1,City Phones: Desk Phones,"\n[INSTRUCTION]\nDraft the section titled ""Cit..."
...,...,...,...,...
1284,CL_0007-RAM-v1.0,CL_0007-RAM-v1.0:S3.5.3,Risk Transfer,"\n[INSTRUCTION]\nDraft the section titled ""Ris..."
1285,CL_0007-RAM-v1.0,CL_0007-RAM-v1.0:S3.6,Risk Assessment Cadence,"\n[INSTRUCTION]\nDraft the section titled ""Ris..."
1286,CL_0007-RAM-v1.0,CL_0007-RAM-v1.0:S3.7,Training and Awareness,"\n[INSTRUCTION]\nDraft the section titled ""Tra..."
1287,CL_0007-RAM-v1.0,CL_0007-RAM-v1.0:S3.8,Risk Treatment,"\n[INSTRUCTION]\nDraft the section titled ""Ris..."


In [17]:
def generate_v2_5_prompt(
    atomic_requirements: List[str],
    security_ontology: List[str],
    kg_context: str,
    company_ontology: Dict[str, str],
    audience: str = "all employees and contractors",
    section_title: str = "",
    policy_title: str = "",
) -> str:

    # -------- ATOMIC REQUIREMENTS --------
    atomic_reqs_text = "\n".join(f"- {req}" for req in atomic_requirements)

    # -------- SECURITY ONTOLOGY BLOCK --------
    if security_ontology and len(security_ontology) > 0:
        security_ontology_text = "\n".join(f"- {concept}" for concept in security_ontology)
        security_ontology_block = f"""[SECURITY ONTOLOGY CONTEXT]
Below are the key security concepts relevant to this section.
Use them only to shape meaning—not as text to copy.

{security_ontology_text}"""
    else:
        security_ontology_block = ""

    # -------- COMPANY ONTOLOGY BLOCK --------
    if company_ontology and len(company_ontology) > 0:
        company_ontology_txt = "\n".join(
            f"- {label}: {value}" for label, value in company_ontology.items()
        )
        company_ontology_block = f"""[COMPANY CONTEXT]
These details describe the organizational environment in which this policy operates.
Use them only to ensure the policy aligns with the organization's characteristics.
Do not restate them verbatim.

{company_ontology_txt}"""
    else:
        company_ontology_block = ""

    # -------- KG CONTEXT BLOCK --------
    if kg_context and len(kg_context.strip()) > 0:
        kg_block = f"""[KNOWLEDGE-GRAPH CONTEXT]
Use the following structural mapping only to maintain conceptual alignment
with the intended control area and sub-control.

{kg_context}"""
    else:
        kg_block = ""

    # -------- ASSEMBLE FINAL PROMPT --------
    v2_5_prompt = f"""
[INSTRUCTION]
Draft the section titled "{section_title}" for the {policy_title} policy.
Audience: {audience}.
Return only the final policy text (no headings, labels, or meta-text).

{security_ontology_block}

{kg_block}

[ATOMIC CONTROL REQUIREMENTS]
These atomic requirements represent the precise expectations this policy section must implement.
They form the minimum set of obligations the text must operationalize.

{atomic_reqs_text}

{company_ontology_block}

[CONSTRAINTS]
- Do not include section numbers or titles.
- Do not mention frameworks, control identifiers, or ontology labels.
- Do not include commentary, explanations, or meta text.
- Integrate overlapping atomic requirements into cohesive, enforceable policy language.
- Align tightly with the atomic requirements, security ontology, and KG mapping.
- Write the final text as if it will appear directly in the approved policy.
""".strip()

    return v2_5_prompt


In [18]:
def get_security_ontology_for_section_from_db(engine, policy_id, section_id, top_k=5, min_sim=0.5):
    query = """
    SELECT ontology_label, similarity
    FROM ontology_alignment
    WHERE level = 'section'
      AND policy_id = %(policy_id)s
      AND section_id = %(section_id)s
      AND similarity >= %(min_sim)s
    ORDER BY similarity DESC
    LIMIT %(top_k)s;
    """
    df = pd.read_sql(query, engine, params={
        "policy_id": policy_id,
        "section_id": section_id,
        "min_sim": min_sim,
        "top_k": top_k,
    })
    return df["ontology_label"].tolist()

In [19]:
# Neo4j Framework Context Extraction
# ============================================================


CYPHER_POLICY_FRAMEWORK_MAP = """
MATCH (f:Framework)
WHERE f.name = $framework_name
MATCH (f)-[:HAS_CONTROL_AREA]->(ca:ControlArea)
MATCH (ca)-[:HAS_SUBCONTROL]->(sc:SubControl)
MATCH (sc)-[:REQUIRES_EVIDENCE]->(er:EvidenceRequirement)
MATCH (p:Policy)-[:SATISFIES_REQUIREMENT]->(er)
WHERE p.title = $policy_name
WITH f, p, ca, sc, er,
     apoc.text.regexGroups(sc.title, ".*\\(([^)]+)\\)") AS groups
RETURN DISTINCT
  f.name              AS framework_name,
  p.title             AS policy_title,
  ca.name             AS control_name,
  ca.code             AS control_id,          // IF you have this
  ca.description      AS control_description, // OR whatever your property is
  sc.title            AS subcontrol_name,
  groups[0][1] AS subcontrol_id,
apoc.text.replace(sc.description, '<[^>]+>', '')   AS subcontrol_description,  // OR sc.text / sc.requirement_text
  er.name            AS evidence_title,
  trim(replace(
    er.example_raw,
    "(document name and content will vary by organization):",
    ""
))     AS evidence_description     // OR er.text / er.requirement
ORDER BY control_name, subcontrol_name


"""

from collections import defaultdict


def get_framework_context_for_policy(policy_name: str,
                                     framework_name: str = "NIST CSF") -> dict:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
    try:
        with driver.session(database=NEO4J_DATABASE) as session:
            rows = session.run(
                CYPHER_POLICY_FRAMEWORK_MAP,
                framework_name=framework_name,
                policy_name=policy_name,
            ).data()
    finally:
        driver.close()

    # Group by control area
    by_control: dict[str, dict] = {}

    for r in rows:
        ctrl_name = r["control_name"]

        if ctrl_name not in by_control:
            by_control[ctrl_name] = {
                "control_area": ctrl_name,
                "control_id": r.get("control_id"),
                "control_description": r.get("control_description"),
                "subcontrols": []
            }

        by_control[ctrl_name]["subcontrols"].append({
            "id": r.get("subcontrol_id"),
            "name": r["subcontrol_name"],
            "description": r.get("subcontrol_description"),
            "evidence_title": r.get("evidence_title"),
            "evidence_description": r.get("evidence_description"),
        })

    return {
        "framework": framework_name,
        "policy": policy_name,
        "controls": list(by_control.values())
    }


In [20]:
def render_compact_kg_context(framework_data: dict,
                              section_title: str,
                              subcontrol_ids: list[str] | None = None,
                              max_controls: int = 3,
                              max_subcontrols: int = 4) -> str:
    """
    Make a short, human-readable KG mapping snippet for the prompt.
    - Optionally filter to a subset of subcontrol IDs relevant to this section.
    """
    fw_name = framework_data.get("framework", "Framework")
    policy_title = framework_data.get("policy", "Policy")

    lines = []

    # 1) Framework + policy line
    lines.append(f"{fw_name} → Policy: {policy_title} → Section: {section_title}")

    # 2) Control-area + subcontrol lines
    controls = framework_data.get("controls", [])

    # Optionally filter by subcontrol_id
    if subcontrol_ids:
        subcontrol_ids_set = set(subcontrol_ids)
        filtered_controls = []
        for c in controls:
            filtered_sub = [
                sc for sc in c.get("subcontrols", [])
                if sc.get("id") in subcontrol_ids_set
            ]
            if filtered_sub:
                c2 = c.copy()
                c2["subcontrols"] = filtered_sub
                filtered_controls.append(c2)
        controls = filtered_controls

    # limit number of controls for brevity
    controls = controls[:max_controls]

    for c in controls:
        ctrl_label = c.get("control_id") or c.get("control_area") or "Control"
        subcontrols = c.get("subcontrols", [])[:max_subcontrols]
        sub_ids = [sc.get("id") or sc.get("name") for sc in subcontrols if sc]
        if not sub_ids:
            continue
        lines.append(f"{ctrl_label} → Subcontrols: {', '.join(sub_ids)}")

    return "\n".join(lines)


In [21]:
def get_kg_context(
    policy_title: str,
    framework: str,
    section_title: str,
    subcontrol_ids: list[str] | None = None,
) -> str:
    framework_data = get_framework_context_for_policy(
        policy_name=policy_title,
        framework_name=framework,
    )
    kg_text = render_compact_kg_context(
        framework_data,
        section_title=section_title,
        subcontrol_ids=subcontrol_ids,
    )
    return kg_text


In [22]:
def build_v2_5_prompt_df(
    section_atomic_with_onto_df,
    allowed_keys,
    audience="all employees and contractors",
    label_lookup=None,
):
    """
    Build a DataFrame with one V2.5 prompt per (policy_id, section_id).
    Mirrors the structure of build_v1_5_prompt_df but includes:
      - security ontology (ontology→atomic alignment)
      - compact KG context block
      - atomic requirements
      - company ontology
    """

    if label_lookup is None:
        label_lookup = {}


    # Sort similar to v1.5: direct first, then inferred, then atomic order
    df_sorted = (
        section_atomic_with_onto_df
        .sort_values(
            ["policy_id", "section_id", "is_direct", "atomic_order"],
            ascending=[True, True, False, True]
        )
    )

    prompt_rows = []

    for (policy_id, section_id), group in df_sorted.groupby(["policy_id", "section_id"]):

        section_title = group["section_title"].iloc[0]
        policy_title  = group["policy_title"].iloc[0]
        framework     = group["source_framework"].iloc[0]

        subcontrol_ids = (
            group["subcontrol_id"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )


        # ------------------------------
        # 1. Atomic requirements
        # ------------------------------
        atomic_requirements = group["atomic_requirement"].tolist()

        # ------------------------------
        # 2. Company ontology
        # ------------------------------
        row0 = group.iloc[0]
        company_ontology = build_company_ontology(row0, allowed_keys, label_lookup)

        # ------------------------------
        # 3. Security ontology concepts (from CyberBERT+, top-k)
        # ------------------------------
        security_ontology = get_security_ontology_for_section_from_db(engine, policy_id, section_id, top_k=5, min_sim=0.5)

        # ------------------------------
        # 4. KG context (compact form)
        # ------------------------------
        kg_context = get_kg_context(
            policy_title=policy_title,
            framework=framework,
            section_title=section_title,
            subcontrol_ids=subcontrol_ids,
        )

        # ------------------------------
        # 5. Assemble V2.5 prompt
        # ------------------------------
        prompt = generate_v2_5_prompt(
            atomic_requirements=atomic_requirements,
            security_ontology=security_ontology,
            kg_context=kg_context,
            company_ontology=company_ontology,
            audience=audience,
            section_title=section_title,
            policy_title=policy_title,
        )

        prompt_rows.append({
            "policy_id": policy_id,
            "section_id": section_id,
            "section_title": section_title,
            "v2_5_prompt": prompt,
        })

    return pd.DataFrame(prompt_rows)


In [23]:
v2_5_prompt_df = build_v2_5_prompt_df(
    section_atomic_with_onto_df,
    allowed_keys=allowed,
    label_lookup=ONTOLOGY_LABELS,
)

In [24]:
v1_system_prompt = """
You are a cybersecurity policy-writing assistant. Your task is to write clear, enforceable,
professionally written policy sections. Each response must align with the provided atomic
requirements and reflect the organization’s environment when client context is supplied.

Write with a formal, directive tone suitable for inclusion in an approved corporate policy.

You must NOT include:
- section titles or numbers
- headings, labels, or bullet lists unless the input explicitly asks for them
- references to frameworks, control identifiers, or ontology fields
- explanations, rationale, or meta-commentary
- references to the instructions themselves

Your output should be a polished policy section that directly operationalizes the provided
atomic requirements and fits naturally into the stated policy.
Return only the final policy text.
"""

v2_system_prompt = """
You are a cybersecurity policy-generation model. Your job is to create precise, enforceable,
ontology-aligned policy sections grounded strictly in the provided inputs.

Each output must reflect:
1. The atomic control requirements
2. The relevant security ontology concepts
3. The knowledge-graph structure linking framework → control area → subcontrol → section
4. The organizational context when available

Use a direct, professional tone consistent with formal IT and cybersecurity governance policies.

You must NOT:
- include section titles, numbers, or headings
- reference any frameworks, control IDs, or ontology fields by name
- restate the company context or ontology verbatim
- provide explanations, reasoning, or meta-text
- add new controls or invent content beyond what is supported by the atomic requirements

Your output must integrate all relevant requirements into a coherent, enforceable policy section,
appropriate for inclusion in a finalized policy document.

Return only the final policy text.
"""

In [25]:
v1_5_save = v1_5_prompt_df.copy()
v1_5_save.rename(columns={"v1_5_prompt": "user_prompt"}, inplace=True)
v1_5_save['version'] = 'v1.5'
v1_5_save['system_prompt'] = v1_system_prompt
v2_5_save = v2_5_prompt_df.copy()
v2_5_save.rename(columns={"v2_5_prompt": "user_prompt"}, inplace=True
)
v2_5_save['version'] = 'v2.5'
v2_5_save['system_prompt'] = v2_system_prompt
v1_5_save.to_sql("policy_prompts", engine, if_exists="replace", index=False )
v2_5_save.to_sql("policy_prompts", engine, if_exists="append", index=False )

289